## 01 — Series Metadata Verification (Pre-Ingestion Check)

Before building the Bronze ingestion pipeline, we verify the actual data
availability for each of the 27 selected series using the Bank of Canada
Valet API metadata endpoint. For each series we retrieve:

- Series label and description (confirmation against our data dictionary)
- First and last observation date (data range currently available)

This step confirms whether historical depth is sufficient for the three
planned analyses (inflation forecasting, yield curve inversion detection,
banking spread anomaly detection) before writing any ingestion code.

In [5]:
import pandas as pd
from pathlib import Path

selected_path = "../discovery/outputs/selected_series.csv"
selected_df = pd.read_csv(selected_path)

print(f"Loaded {selected_df.shape[0]} selected series")
selected_df.head()

Loaded 27 selected series


,series_id,domain,justification,label,link
0,CBC20210,Interest Rates,Target for the overnight rate (Bank of Canada ...,V39079,https://www.bankofcanada.ca/valet/series/CBC20210
1,BD.CDN.2YR.DQ.YLD,Fixed Income,Benchmark bond yield: 2 year,Benchmark bond yield: 2 year,https://www.bankofcanada.ca/valet/series/BD.CD...
2,BD.CDN.3YR.DQ.YLD,Fixed Income,Benchmark bond yield: 3 year,Benchmark bond yield: 3 year,https://www.bankofcanada.ca/valet/series/BD.CD...
3,BD.CDN.5YR.DQ.YLD,Fixed Income,Benchmark bond yield: 5 year,Benchmark bond yield: 5 year,https://www.bankofcanada.ca/valet/series/BD.CD...
4,BD.CDN.7YR.DQ.YLD,Fixed Income,Benchmark bond yield: 7 year,Benchmark bond yield: 7 year,https://www.bankofcanada.ca/valet/series/BD.CD...


In [ ]:
import requests

VALET_BASE_URL = "https://www.bankofcanada.ca/valet"


def get_series_detail(series_id):
    """Fetch the metadata of one series from the Valet `/series/{id}/json` endpoint."""
    url = f"{VALET_BASE_URL}/series/{series_id}/json"
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return response.json()

In [6]:
# Inspect the actual top-level structure of the response
test = get_series_detail("CBC20210")
print(test.keys())
test

dict_keys(['terms', 'seriesDetails'])


{'terms': {'url': 'https://www.bankofcanada.ca/terms/'},
 'seriesDetails': {'name': 'CBC20210',
  'label': 'V39079',
  'description': 'Target for the overnight rate'}}

## 01.1 — Corrected API Response Structure

The series metadata endpoint (`/series/{id}/json`) returns two top-level
keys:

- `terms`: link to the terms of use
- `seriesDetails`: a flat object with `name`, `label`, `description`
  (no nested key by series_id, unlike initially assumed)

This endpoint does not include the observation date range. To retrieve
first/last observation dates, we need the observations endpoint
(`/observations/{id}/json`), requesting a minimal date range to inspect
its structure before looping over all 27 series.

In [7]:
def get_observations_sample(series_id):
    """Fetch a minimal observations response to inspect structure and date range."""
    url = f"https://www.bankofcanada.ca/valet/observations/{series_id}/json"
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return response.json()

obs_test = get_observations_sample("CBC20210")
print(obs_test.keys())
print(obs_test.get("seriesDetail", obs_test.get("seriesDetails")))
print(obs_test["observations"][:3])   # first 3 points
print(obs_test["observations"][-3:])  # last 3 points
print(len(obs_test["observations"]))

dict_keys(['terms', 'seriesDetail', 'observations'])
{'CBC20210': {'label': 'V39079', 'description': 'Target for the overnight rate', 'dimension': {'key': 'd', 'name': 'Date'}}}
[{'d': '2001-01-01', 'CBC20210': {'v': '5.75'}}, {'d': '2001-01-02', 'CBC20210': {'v': '5.75'}}, {'d': '2001-01-03', 'CBC20210': {'v': '5.75'}}]
[{'d': '2026-07-14', 'CBC20210': {'v': '2.25'}}, {'d': '2026-07-15', 'CBC20210': {'v': '2.25'}}, {'d': '2026-07-20', 'CBC20210': {'v': '2.25'}}]
6651


## 01.2 — Full Metadata Verification Loop (27 Selected Series)

We now loop over all 27 selected series to extract, for each one:

- First observation date
- Last observation date
- Total number of observations
- Implied average frequency (based on date span / count)

Note: the observations endpoint uses the key `seriesDetail` (singular),
while the series metadata endpoint uses `seriesDetails` (plural) — this
inconsistency is handled explicitly in the code below to avoid silent
KeyErrors.

A short delay is added between requests to respect the API's terms of use
(avoid overloading the server).

In [8]:
import time

def get_series_coverage(series_id):
    """Fetch observation date range and count for a single series."""
    url = f"https://www.bankofcanada.ca/valet/observations/{series_id}/json"
    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        data = response.json()

        observations = data["observations"]
        n_obs = len(observations)

        if n_obs == 0:
            return {
                "series_id": series_id,
                "status": "empty",
                "first_date": None,
                "last_date": None,
                "n_observations": 0,
            }

        first_date = observations[0]["d"]
        last_date = observations[-1]["d"]

        return {
            "series_id": series_id,
            "status": "ok",
            "first_date": first_date,
            "last_date": last_date,
            "n_observations": n_obs,
        }

    except Exception as e:
        return {
            "series_id": series_id,
            "status": f"error: {e}",
            "first_date": None,
            "last_date": None,
            "n_observations": None,
        }

# Loop over all 27 selected series (from selected_df built earlier)
coverage_results = []
for sid in selected_df["series_id"]:
    result = get_series_coverage(sid)
    coverage_results.append(result)
    time.sleep(0.3)  # polite delay between requests

coverage_df = pd.DataFrame(coverage_results)
coverage_df = coverage_df.merge(selected_df[["series_id", "domain"]], on="series_id", how="left")
coverage_df

,series_id,status,first_date,last_date,n_observations,domain
0,CBC20210,ok,2001-01-01,2026-07-20,6651,Interest Rates
1,BD.CDN.2YR.DQ.YLD,ok,2001-01-02,2026-07-20,6388,Fixed Income
2,BD.CDN.3YR.DQ.YLD,ok,2001-01-02,2026-07-20,6388,Fixed Income
3,BD.CDN.5YR.DQ.YLD,ok,2001-01-02,2026-07-20,6388,Fixed Income
4,BD.CDN.7YR.DQ.YLD,ok,2001-01-02,2026-07-20,6388,Fixed Income
5,BD.CDN.10YR.DQ.YLD,ok,2001-01-02,2026-07-20,6389,Fixed Income
6,BD.CDN.LONG.DQ.YLD,ok,2001-01-02,2026-07-20,6388,Fixed Income
7,TB.CDN.30D.MID,ok,2001-01-02,2026-07-20,6387,Fixed Income
8,TB.CDN.60D.MID,ok,2001-01-02,2026-07-20,6387,Fixed Income
9,TB.CDN.90D.MID,ok,2001-01-02,2026-07-20,6387,Fixed Income


### 01.3 — Coverage Analysis and Implications

All 27 series return valid data ("ok" status), but with significant
differences in historical depth:

- **Monetary aggregates and mortgage rates** offer the longest history
  (back to 1969 and 1975 respectively), covering multiple full economic
  cycles.
- **CPI and core inflation measures** cover roughly 30 years (1993/1995
  onwards).
- **Bond yields and treasury bills** cover 25 years (2001 onwards).
- **Foreign exchange series are limited to 2017 onwards** (~9 years). This
  is a known limitation of the Valet API's daily FX data, not a selection
  issue — the API does not return FX observations before 2017-01-03
  regardless of the requested start date.

**Implication for the planned predictive analyses:**
- Inflation forecasting and banking spread anomaly detection remain fully
  feasible using data from 2001 onwards (25 years, multiple economic
  cycles).
- Yield curve inversion prediction is feasible but will face a
  class-imbalance challenge: few actual inversion episodes exist within
  the available window (likely 2007-08, and potentially 2019 and
  2022-23), despite thousands of daily observations.
- FX-based volatility features should be treated as a recent-period
  signal only (post-2017), since they cannot capture the 2008 financial
  crisis.

These findings will directly inform the date range used for the unified
Silver layer and the train/test split strategy for the predictive models.

## 02 — Data Quality Assessment

Coverage verification (Section 01) confirmed date ranges and observation
counts, but did not inspect the content of each series. Before designing
the Bronze schema, we assess data quality for each of the 27 series:

- Missing or null values (`v` field empty or absent)
- Non-numeric values (parsing errors when converting to float)
- Duplicate dates
- Gaps in the date sequence beyond expected non-business days
- Basic sanity checks on value ranges (e.g., unexpected negative rates
  where none are expected)

This step determines whether any series requires special handling
(imputation, exclusion of bad records, flagging) before ingestion.

In [9]:
import numpy as np

def get_full_observations(series_id):
    """Fetch all observations for a series."""
    url = f"https://www.bankofcanada.ca/valet/observations/{series_id}/json"
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    return response.json()["observations"]

def assess_series_quality(series_id):
    """Run basic data quality checks on a single series."""
    obs = get_full_observations(series_id)

    records = []
    for o in obs:
        raw_value = o.get(series_id, {}).get("v", None)
        records.append({"date": o["d"], "raw_value": raw_value})

    s = pd.DataFrame(records)
    s["date"] = pd.to_datetime(s["date"])

    # Missing / empty values
    n_missing = s["raw_value"].isna().sum() + (s["raw_value"] == "").sum()

    # Non-numeric values (fail float conversion)
    def is_numeric(x):
        try:
            float(x)
            return True
        except (TypeError, ValueError):
            return False

    n_non_numeric = (~s["raw_value"].apply(is_numeric)).sum()

    # Duplicate dates
    n_duplicates = s["date"].duplicated().sum()

    # Gaps: max distance between consecutive observations
    date_diffs = s["date"].sort_values().diff().dt.days
    max_gap_days = date_diffs.max()

    return {
        "series_id": series_id,
        "n_observations": len(s),
        "n_missing": int(n_missing),
        "n_non_numeric": int(n_non_numeric),
        "n_duplicates": int(n_duplicates),
        "max_gap_days": max_gap_days,
    }

# Run on all 27 series
quality_results = []
for sid in selected_df["series_id"]:
    result = assess_series_quality(sid)
    quality_results.append(result)
    time.sleep(0.3)

quality_df = pd.DataFrame(quality_results)
quality_df = quality_df.merge(selected_df[["series_id", "domain"]], on="series_id", how="left")
quality_df

,series_id,n_observations,n_missing,n_non_numeric,n_duplicates,max_gap_days,domain
0,CBC20210,6651,0,0,0,5.0,Interest Rates
1,BD.CDN.2YR.DQ.YLD,6388,0,0,0,5.0,Fixed Income
2,BD.CDN.3YR.DQ.YLD,6388,0,0,0,5.0,Fixed Income
3,BD.CDN.5YR.DQ.YLD,6388,0,0,0,5.0,Fixed Income
4,BD.CDN.7YR.DQ.YLD,6388,0,0,0,5.0,Fixed Income
5,BD.CDN.10YR.DQ.YLD,6389,0,0,0,5.0,Fixed Income
6,BD.CDN.LONG.DQ.YLD,6388,0,0,0,5.0,Fixed Income
7,TB.CDN.30D.MID,6387,0,0,0,5.0,Fixed Income
8,TB.CDN.60D.MID,6387,0,0,0,5.0,Fixed Income
9,TB.CDN.90D.MID,6387,0,0,0,5.0,Fixed Income


## 02.1 — Value Range Sanity Check

Structural quality checks (missing values, non-numeric entries,
duplicates, gaps) all passed cleanly across the 27 series. The `max_gap_days`
results also confirm real publication frequencies:

- 5-day gaps for daily series (long weekends / holidays) — as expected
- 31-day gaps for monthly series (CPI, monetary aggregates) — expected,
  not a data issue
- 7-day gaps for mortgage rate series — reveals these are published
  **weekly**, not daily as initially assumed in the data dictionary. This
  will be corrected.

One check remains: verifying that observed values fall within plausible
ranges for each series (e.g., no impossible negative bond yields, no
implausible single-day jumps), since count-based checks cannot catch this.

In [10]:
def value_range_summary(series_id):
    """Compute basic descriptive statistics on the numeric values of a series."""
    obs = get_full_observations(series_id)
    values = [float(o[series_id]["v"]) for o in obs if series_id in o]

    s = pd.Series(values)
    diffs = s.diff().abs()

    return {
        "series_id": series_id,
        "min": s.min(),
        "max": s.max(),
        "mean": round(s.mean(), 3),
        "max_daily_jump": round(diffs.max(), 3),
    }

range_results = []
for sid in selected_df["series_id"]:
    result = value_range_summary(sid)
    range_results.append(result)
    time.sleep(0.3)

range_df = pd.DataFrame(range_results)
range_df = range_df.merge(selected_df[["series_id", "domain"]], on="series_id", how="left")
range_df

,series_id,min,max,mean,max_daily_jump,domain
0,CBC20210,0.25000,5.75000,2.033,1.000,Interest Rates
1,BD.CDN.2YR.DQ.YLD,0.15000,5.11000,2.244,0.430,Fixed Income
2,BD.CDN.3YR.DQ.YLD,0.18000,5.45000,2.357,0.470,Fixed Income
3,BD.CDN.5YR.DQ.YLD,0.30000,5.63000,2.590,0.330,Fixed Income
4,BD.CDN.7YR.DQ.YLD,0.32000,5.88000,2.765,0.290,Fixed Income
5,BD.CDN.10YR.DQ.YLD,0.43000,5.99000,3.019,0.260,Fixed Income
6,BD.CDN.LONG.DQ.YLD,0.71000,6.11000,3.395,0.280,Fixed Income
7,TB.CDN.30D.MID,0.00000,5.41000,1.881,0.800,Fixed Income
8,TB.CDN.60D.MID,-0.01000,5.44000,1.913,0.800,Fixed Income
9,TB.CDN.90D.MID,0.00000,5.46000,1.943,1.030,Fixed Income


## 03 — Technical Schema Dictionary

Beyond the business/financial data dictionary, we now consolidate a
technical schema reference for each series: exact column structure, data
type, unit, and expected value range. This becomes the schema contract
used to define the Bronze Delta table structure during ingestion.

In [11]:
# Consolidate technical schema metadata per series using previously collected results
schema_dict = coverage_df.merge(
    quality_df.drop(columns=["domain"]), on="series_id", how="left", suffixes=("", "_qual")
).merge(
    range_df.drop(columns=["domain"]), on="series_id", how="left"
)

schema_dict["unit"] = schema_dict["domain"].map({
    "Interest Rates": "percent (%)",
    "Fixed Income": "percent (%)",
    "Foreign Exchange": "CAD per 1 unit of foreign currency",
    "Inflation": "percent (%), year-over-year or month-over-month",
    "Monetary Aggregates": "percent (%) year-over-year growth",
    "Banking": "percent (%)",
})

schema_dict["value_dtype"] = "float64 (raw API value is string, requires casting)"
schema_dict["date_dtype"] = "date (raw API format: YYYY-MM-DD string)"

schema_dict[[
    "series_id", "domain", "unit", "value_dtype", "date_dtype",
    "first_date", "last_date", "n_observations", "min", "max", "max_gap_days"
]]

,series_id,domain,unit,value_dtype,date_dtype,first_date,last_date,n_observations,min,max,max_gap_days
0,CBC20210,Interest Rates,percent (%),"float64 (raw API value is string, requires cas...",date (raw API format: YYYY-MM-DD string),2001-01-01,2026-07-20,6651,0.25000,5.75000,5.0
1,BD.CDN.2YR.DQ.YLD,Fixed Income,percent (%),"float64 (raw API value is string, requires cas...",date (raw API format: YYYY-MM-DD string),2001-01-02,2026-07-20,6388,0.15000,5.11000,5.0
2,BD.CDN.3YR.DQ.YLD,Fixed Income,percent (%),"float64 (raw API value is string, requires cas...",date (raw API format: YYYY-MM-DD string),2001-01-02,2026-07-20,6388,0.18000,5.45000,5.0
3,BD.CDN.5YR.DQ.YLD,Fixed Income,percent (%),"float64 (raw API value is string, requires cas...",date (raw API format: YYYY-MM-DD string),2001-01-02,2026-07-20,6388,0.30000,5.63000,5.0
4,BD.CDN.7YR.DQ.YLD,Fixed Income,percent (%),"float64 (raw API value is string, requires cas...",date (raw API format: YYYY-MM-DD string),2001-01-02,2026-07-20,6388,0.32000,5.88000,5.0
5,BD.CDN.10YR.DQ.YLD,Fixed Income,percent (%),"float64 (raw API value is string, requires cas...",date (raw API format: YYYY-MM-DD string),2001-01-02,2026-07-20,6389,0.43000,5.99000,5.0
6,BD.CDN.LONG.DQ.YLD,Fixed Income,percent (%),"float64 (raw API value is string, requires cas...",date (raw API format: YYYY-MM-DD string),2001-01-02,2026-07-20,6388,0.71000,6.11000,5.0
7,TB.CDN.30D.MID,Fixed Income,percent (%),"float64 (raw API value is string, requires cas...",date (raw API format: YYYY-MM-DD string),2001-01-02,2026-07-20,6387,0.00000,5.41000,5.0
8,TB.CDN.60D.MID,Fixed Income,percent (%),"float64 (raw API value is string, requires cas...",date (raw API format: YYYY-MM-DD string),2001-01-02,2026-07-20,6387,-0.01000,5.44000,5.0
9,TB.CDN.90D.MID,Fixed Income,percent (%),"float64 (raw API value is string, requires cas...",date (raw API format: YYYY-MM-DD string),2001-01-02,2026-07-20,6387,0.00000,5.46000,5.0


## 03.1 — Export Technical Schema Dictionary

The technical schema dictionary is exported as the reference contract for
the Bronze ingestion layer. It consolidates, for each of the 27 series:
business domain, unit, raw data types, date coverage, and value ranges.

This file — together with the business data dictionary (Word document)
and the financial selection rationale — completes the documentation set
for Phase 1 (Data Discovery) before starting Phase 2 (Ingestion).

In [12]:
output_path = Path("../discovery/outputs/technical_schema_dictionary.csv")
schema_dict.to_csv(output_path, index=False)

print(f"Technical schema dictionary exported to: {output_path}")
schema_dict.shape

Technical schema dictionary exported to: ../discovery/outputs/technical_schema_dictionary.csv


(27, 18)